# 🧩 Mini-Lab: Calculator Tool

**Module 7: AI Agents** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Execute** tool calls by dispatching the LLM's requested function with parsed arguments
2. **Extract** parameters from the LLM's tool call response and map them to Python function arguments
3. **Parse** tool results and feed them back to the LLM to generate a final user-facing response

## Target Concepts

| Concept | Description |
|---------|-------------|
| Tool Execution | Invoking the actual Python function when the LLM requests a tool call |
| Parameter Extraction | Parsing the JSON arguments string from the LLM into Python values |
| Response Parsing | Processing the tool's return value and sending it back to the LLM for a final answer |

## Setup

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # Uses OPENAI_API_KEY from .env

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Recap: From Definition to Execution

In the previous mini-lab (`mini-function-call`), we learned how to **define** tools and saw the full round trip. In this lab, we zoom in on the three steps that happen *after* the LLM returns a tool call:

1. **Parameter Extraction** — parsing the raw JSON arguments string into Python values
2. **Tool Execution** — dispatching the right function with those parsed arguments
3. **Response Parsing** — packaging the function's output and sending it back for a final answer

We'll build a **calculator tool** to practice these concepts with multiple operations and parameter types.

## Step 1: Define the Calculator Function and Tool Schema

Our calculator supports four operations. We define both the Python function and the tool schema the LLM needs.

In [3]:
def calculate(operation, a, b):
    """Perform a math operation on two numbers."""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: division by zero"
    }
    result = ops.get(operation, "Error: unknown operation")
    return json.dumps({"operation": operation, "a": a, "b": b, "result": result})


tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a math operation (add, subtract, multiply, divide) on two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The math operation to perform."
                    },
                    "a": {
                        "type": "number",
                        "description": "The first number."
                    },
                    "b": {
                        "type": "number",
                        "description": "The second number."
                    }
                },
                "required": ["operation", "a", "b"]
            }
        }
    }
]

# Function registry — maps names to callables
available_functions = {
    "calculate": calculate
}

print("Calculator tool defined with operations: add, subtract, multiply, divide")

Calculator tool defined with operations: add, subtract, multiply, divide


## Step 2: Get a Tool Call from the LLM

Let's send a math question and inspect the raw tool call the LLM returns. We'll pause here to examine what we get before executing anything.

In [4]:
messages = [{"role": "user", "content": "What is 147 multiplied by 83?"}]

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
tool_call = assistant_message.tool_calls[0]

print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Function name: {tool_call.function.name}")
print(f"Raw arguments: {tool_call.function.arguments}")
print(f"Argument type: {type(tool_call.function.arguments)}")

Finish reason: tool_calls
Function name: calculate
Raw arguments: {"operation":"multiply","a":147,"b":83}
Argument type: <class 'str'>


Notice that `tool_call.function.arguments` is a **string**, not a dictionary. The LLM outputs JSON text — we need to parse it ourselves. This is the **parameter extraction** step.

## Step 3: Parameter Extraction

We use `json.loads()` to convert the raw argument string into a Python dictionary. Let's inspect each extracted parameter and its type.

In [5]:
# Extract parameters from the tool call
raw_args = tool_call.function.arguments
parsed_args = json.loads(raw_args)

md(f"""
### Extracted Parameters

| Parameter | Value | Python Type |
|-----------|-------|-------------|
| `operation` | `{parsed_args['operation']}` | `{type(parsed_args['operation']).__name__}` |
| `a` | `{parsed_args['a']}` | `{type(parsed_args['a']).__name__}` |
| `b` | `{parsed_args['b']}` | `{type(parsed_args['b']).__name__}` |
""")


### Extracted Parameters

| Parameter | Value | Python Type |
|-----------|-------|-------------|
| `operation` | `multiply` | `str` |
| `a` | `147` | `int` |
| `b` | `83` | `int` |


The LLM translated the natural language *"147 multiplied by 83"* into structured parameters:
- It chose `"multiply"` from the allowed enum values
- It extracted `147` and `83` as numbers (not strings)

This is the power of parameter extraction — the LLM does the hard work of parsing natural language into typed arguments.

## Step 4: Tool Execution

Now we **dispatch** the call — look up the function by name and invoke it with the extracted arguments.

In [6]:
# Look up the function by name
function_name = tool_call.function.name
function_to_call = available_functions[function_name]

# Execute with the parsed arguments
result = function_to_call(**parsed_args)

print(f"Function called: {function_name}")
print(f"Arguments:       {parsed_args}")
print(f"Raw result:      {result}")
print(f"Result type:     {type(result).__name__}")

Function called: calculate
Arguments:       {'operation': 'multiply', 'a': 147, 'b': 83}
Raw result:      {"operation": "multiply", "a": 147, "b": 83, "result": 12201}
Result type:     str


Key points about tool execution:
- We use a **function registry** (`available_functions` dict) to map the name string to the actual callable
- We use `**parsed_args` to unpack the dictionary as keyword arguments
- The result is a JSON string — tool results sent back to the LLM must always be strings

## Step 5: Response Parsing — Sending Results Back

The final step is **response parsing**: we package the tool result into the conversation and let the LLM generate a human-friendly answer.

In [7]:
# Build the complete message history with the tool result
messages = [
    {"role": "user", "content": "What is 147 multiplied by 83?"},
    assistant_message,  # Contains the tool_calls request
    {
        "role": "tool",
        "tool_call_id": tool_call.id,  # Must match the original tool call
        "content": result                # Must be a string
    }
]

# LLM reads the tool result and generates a final answer
final_response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools
)

md(f"**Assistant:** {final_response.choices[0].message.content}")

**Assistant:** 147 multiplied by 83 equals 12,201.

The tool message requires two critical fields:
- **`tool_call_id`** — links this result to the specific tool call that requested it
- **`content`** — the result string the LLM will read to formulate its answer

The LLM takes the raw JSON result (`{"result": 12201}`) and turns it into a natural response.

## Putting It Together: Reusable Helper

Let's combine all three steps into a clean helper that handles parameter extraction → tool execution → response parsing.

In [8]:
def execute_tool_calls(assistant_msg):
    """Extract parameters, execute tools, return results as messages."""
    tool_messages = []
    
    for tc in assistant_msg.tool_calls:
        # Parameter extraction
        args = json.loads(tc.function.arguments)
        
        # Tool execution
        fn = available_functions[tc.function.name]
        result = fn(**args)
        
        # Response parsing — package for the LLM
        tool_messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result
        })
        
        print(f"  ⚙️ {tc.function.name}({args}) → {result}")
    
    return tool_messages

In [9]:
def ask_calculator(question):
    """Ask a math question using the calculator tool."""
    messages = [{"role": "user", "content": question}]
    
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        tools=tools
    )
    
    assistant_msg = response.choices[0].message
    
    if assistant_msg.tool_calls:
        messages.append(assistant_msg)
        messages.extend(execute_tool_calls(assistant_msg))
        
        final = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages,
            tools=tools
        )
        return final.choices[0].message.content
    
    return assistant_msg.content

In [10]:
# Test with different operations
questions = [
    "What is 256 divided by 8?",
    "Subtract 37 from 100",
    "Add 1999 and 2024",
]

for q in questions:
    md(f"**Q:** {q}")
    answer = ask_calculator(q)
    md(f"**A:** {answer}\n\n---")

**Q:** What is 256 divided by 8?

  ⚙️ calculate({'operation': 'divide', 'a': 256, 'b': 8}) → {"operation": "divide", "a": 256, "b": 8, "result": 32.0}


**A:** 256 divided by 8 is 32.

---

**Q:** Subtract 37 from 100

  ⚙️ calculate({'operation': 'subtract', 'a': 100, 'b': 37}) → {"operation": "subtract", "a": 100, "b": 37, "result": 63}


**A:** 100 minus 37 equals 63.

---

**Q:** Add 1999 and 2024

  ⚙️ calculate({'operation': 'add', 'a': 1999, 'b': 2024}) → {"operation": "add", "a": 1999, "b": 2024, "result": 4023}


**A:** The sum of 1999 and 2024 is 4023.

---

For each question, the LLM:
1. **Extracted** the right operation and numbers from natural language
2. Our code **executed** the function with those parameters
3. The LLM **parsed** the raw result into a natural-language answer

## Edge Case: Division by Zero

Let's see how our tool handles an error and how the LLM responds to the error message.

In [11]:
answer = ask_calculator("What is 42 divided by 0?")
md(f"**A:** {answer}")

**A:** Dividing a number by zero is undefined in mathematics. It does not have a meaning within the real number system. Would you like me to explain why or assist you with a different calculation?

The tool returned an error string, and the LLM gracefully communicated it to the user. The result string doesn't have to be a number — it can be any information, including error messages.

## 🎯 Summary

### Key Takeaways

1. **Parameter extraction** — use `json.loads()` to parse the LLM's argument string into a Python dictionary with properly typed values
2. **Tool execution** — use a function registry dict to look up the function by name and `**kwargs` to unpack the parsed arguments
3. **Response parsing** — send the tool result back as a `"role": "tool"` message with the matching `tool_call_id`, and the LLM converts raw output into a natural answer
4. **Results must be strings** — tool results sent back to the LLM must always be serialized as strings (use `json.dumps()`)
5. **Error handling in results** — tools can return error messages as strings, and the LLM will communicate them gracefully to the user